In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Import Model
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Pastikan library 'openpyxl' sudah terinstall: pip install openpyxl pandas scikit-learn
file_path = "Data Collection - Final Project DSML (1).xlsx"  # Ganti nama file sesuai dengan nama file Excel Anda

# Jika data berada di sheet tertentu, tambahkan argument: sheet_name='Sheet1'
df = pd.read_excel(file_path)

# Menampilkan 5 baris pertama data untuk memastikan berhasil terbaca
print("=== 5 Data Pertama ===")
print(df.head())

=== 5 Data Pertama ===
   No.    Ukuran Kelayakan (Grade)         Kondisi Tipe Kain  Tipe Baju  \
0    1         M                 B  Ada noda kecil      Katun      Kaos   
1    2  All Size                 A               -      Rayon     Kerah   
2    3  All Size                 A  Kancing beda 1  Polyester     Kerah   
3    4         M                 B               -      Katun      Kaos   
4    5         M                 B               -      Katun      Kaos   

     Merk (Merk/Non-Merk Polosan/Gambar (Y/N) Gender (M/F/U)  Harga  ...  \
0  Merk (The Excecutive)               Gambar              F  45000  ...   
1               Non-Merk                Polos              F  35000  ...   
2          Merk (Avenue)                Polos              F  55000  ...   
3        Merk (Giordano)                Polos              F  45000  ...   
4        Merk (Giordano)                Polos              F  45000  ...   

   Grade  Unnamed: 14 Unnamed: 15 Unnamed: 16 Unnamed: 17 Unnamed: 18

In [7]:
# ==========================================
# 2. PEMBERSIHAN DATA (DATA CLEANING)
# ==========================================
# Fungsi untuk membersihkan format harga (misal: "IDR 40.000,00" -> 40000.0)
def clean_currency_val(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    # Bersihkan karakter teks 'IDR', titik ribuan, dan ganti koma desimal
    s = str(val).replace('IDR', '').strip()
    s = s.replace('.', '').replace(',', '.')
    return float(s)

# Menghilangkan spasi berlebih pada nama-nama kolom agar tidak rentan error
df.columns = df.columns.str.strip()

# Bersihkan kolom target (berdasarkan data Anda, kolom harga bernama 'Harga')
target_col = 'Harga'
df[target_col] = df[target_col].apply(clean_currency_val)

# Pilih fitur-fitur kategorikal berdasarkan kolom riil di Excel Anda setelah di-strip
feature_cols = [
    'Ukuran',
    'Kelayakan (Grade)',
    'Kondisi',
    'Tipe Kain',
    'Tipe Baju',
    'Merk (Merk/Non-Merk',
    'Polosan/Gambar (Y/N)',
    'Gender (M/F/U)'
]

X = df[feature_cols]
y = df[target_col]

In [8]:
# ==========================================
# 3. PREPROCESSING PIPELINE
# ==========================================
categorical_features = feature_cols

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Tidak Ada')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [9]:
# ==========================================
# 4. SPLIT DATA
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# ==========================================
# 5. DEFINISI MODEL (BASELINE & MODEL UTAMA)
# ==========================================
models = {
    "Baseline 1: Linear Regression": LinearRegression(),
    "Baseline 2: Decision Tree": DecisionTreeRegressor(random_state=42),
    "Model Utama: Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

In [10]:
# ==========================================
# 6. TRAINING & EVALUASI
# ==========================================
results = []

for name, model in models.items():
    # Buat Pipeline
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Latih Model
    clf.fit(X_train, y_train)

    # Prediksi
    y_pred = clf.predict(X_test)

    # Hitung Metrik Evaluasi
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE (IDR)": round(mae, 2),
        "RMSE (IDR)": round(rmse, 2),
        "R2 Score": round(r2, 4)
    })

# ==========================================
# 7. TAMPILKAN HASIL EVALUASI
# ==========================================
df_results = pd.DataFrame(results)
print("\n=== HASIL PERBANDINGAN MODEL ===")
print(df_results.to_string(index=False))


=== HASIL PERBANDINGAN MODEL ===
                        Model  MAE (IDR)  RMSE (IDR)  R2 Score
Baseline 1: Linear Regression   19083.54    25357.75    0.2287
    Baseline 2: Decision Tree   17482.28    28599.99    0.0188
   Model Utama: Random Forest   13024.72    19038.40    0.5652


In [11]:
import joblib
import os

# 1. Definisikan ulang dan latih pipeline model terbaik (Random Forest)
best_model_name = "Model Utama: Random Forest"
best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', models[best_model_name])
])

# Latih kembali dengan data training
best_pipeline.fit(X_train, y_train)

# 2. Tentukan path penyimpanan di Google Drive Anda
drive_path = "/content/drive/MyDrive/"
model_filename = "random_forest_model.joblib"
full_save_path = os.path.join(drive_path, model_filename)

# 3. Simpan model menggunakan joblib
try:
    joblib.dump(best_pipeline, full_save_path)
    print(f"Model berhasil disimpan di: {full_save_path}")
except Exception as e:
    print(f"Gagal menyimpan ke Drive. Menyimpan secara lokal di runtime saja... Error: {e}")
    joblib.dump(best_pipeline, model_filename)
    print(f"Model berhasil disimpan secara lokal sebagai: {model_filename}")

Model berhasil disimpan di: /content/drive/MyDrive/random_forest_model.joblib
